Install Packages

In [1]:
!pip install -q pymupdf
!pip install -q chromadb
!pip install -q sentence-transformers
!pip install -q google-generativeai
!pip install -q gradio
!pip install -q langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 47.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

Imports

In [2]:
import fitz
import chromadb
import gradio as gr
import google.generativeai as genai

from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("All Imports Loaded Successfully")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


All Imports Loaded Successfully


Gemini Setup

In [3]:
genai.configure(
    api_key="YOUR_GEMINI_API_KEY"
)

chat_model = genai.GenerativeModel(
    "gemini-2.5-flash"
)

print("Gemini Ready")

Gemini Ready


Embedding Model

In [4]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Ready


ChromaDB Setup

In [5]:
client = chromadb.Client()

collection = None

print("ChromaDB Ready")

ChromaDB Ready


Create PDF Processing Function

In [6]:
def process_pdfs(files):

    global collection

    client = chromadb.Client()

    try:
        client.delete_collection("pdf_collection")
    except:
        pass

    collection = client.create_collection(
        name="pdf_collection"
    )

    full_text = ""

    for file in files:

        pdf = fitz.open(file.name)

        for page in pdf:
            full_text += page.get_text()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_text(full_text)

    embeddings = embedding_model.encode(chunks)

    for i, chunk in enumerate(chunks):

        collection.add(
            ids=[str(i)],
            documents=[chunk],
            embeddings=[embeddings[i].tolist()]
        )

    return f"Successfully processed {len(files)} PDF(s)"

process pdfs

In [7]:
print(process_pdfs)

<function process_pdfs at 0x7c9879071ee0>


Question Answering Function

In [8]:
def ask_question(question):

    global collection

    if collection is None:
        return "Please upload and process PDFs first."

    query_embedding = embedding_model.encode(question)

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=5
    )

    context = "\n\n".join(
        results["documents"][0]
    )

    prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}
"""

    response = chat_model.generate_content(
        prompt
    )

    return response.text

Test Function Exists

In [9]:
print(ask_question)

<function ask_question at 0x7c98790728e0>


Multi-PDF Gradio UI

In [10]:
with gr.Blocks() as demo:

    gr.Markdown("# 📚 Multi-PDF RAG Chatbot")

    gr.Markdown(
        "Upload multiple PDFs, process them, and ask questions."
    )

    pdf_files = gr.File(
        file_count="multiple",
        file_types=[".pdf"],
        label="Upload PDFs"
    )

    process_button = gr.Button(
        "Process PDFs"
    )

    status_box = gr.Textbox(
        label="Status"
    )

    process_button.click(
        fn=process_pdfs,
        inputs=pdf_files,
        outputs=status_box
    )

    question_box = gr.Textbox(
        label="Ask a Question",
        placeholder="Example: What is process management?"
    )

    ask_button = gr.Button(
        "Ask Question"
    )

    answer_box = gr.Textbox(
        label="Answer",
        lines=10
    )

    ask_button.click(
        fn=ask_question,
        inputs=question_box,
        outputs=answer_box
    )

demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7a8fd7a57131e7fcd2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


MuPDF error: format error: No default Layer config

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7a8fd7a57131e7fcd2.gradio.live
